# Roland-Gamos — Pipeline de génération d'assets (pixel art)

Pipeline auto-hébergé sur **Google Colab (GPU gratuit T4)** : Stable Diffusion XL + LoRA [`nerijs/pixel-art-xl`](https://huggingface.co/nerijs/pixel-art-xl) + LoRA LCM pour un rendu rapide (8 étapes, ~5-10s/image sur T4).

**Avant de lancer :** menu `Exécution > Modifier le type d'exécution` → GPU (T4 gratuit).

Ce notebook fait, dans l'ordre :
1. Monte ton Google Drive (stockage persistant entre sessions — le cache du modèle et les images générées y restent, donc pas de retéléchargement de ~7 Go à chaque reconnexion).
2. Installe les dépendances.
3. Charge SDXL + LoRA pixel art + LoRA LCM.
4. Génère un **petit lot de test (3 avatars)** pour valider le style avant de lancer la génération complète (~600 assets, dans un notebook/manifest séparé une fois ce test validé).

**Limite du Colab gratuit** : session coupée après ~12h ou après une période d'inactivité, et disponibilité GPU non garantie instantanément. Le cache modèle sur Drive limite la casse en cas de reconnexion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/RolandGamosAssets'
CACHE = f'{BASE}/hf_cache'
OUTPUT = f'{BASE}/output'
for sub in ['avatars', 'skins', 'auras', 'cadres', 'effets', 'animations', 'test']:
    os.makedirs(f'{OUTPUT}/{sub}', exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
print('Dossiers prêts dans', BASE)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors peft rembg onnxruntime

In [ ]:
import os
os.environ['HF_HOME'] = CACHE
os.environ['HF_HUB_CACHE'] = CACHE

import torch
from diffusers import DiffusionPipeline, LCMScheduler, AutoencoderKL

print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')

In [ ]:
# VAE fixe recommandé par nerijs pour éviter les artefacts avec ce LoRA
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',
    torch_dtype=torch.float16,
    cache_dir=CACHE,
)

pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    vae=vae,
    torch_dtype=torch.float16,
    variant='fp16',
    cache_dir=CACHE,
).to('cuda')

# LoRA pixel art (nerijs) — force recommandée 1.2
pipe.load_lora_weights('nerijs/pixel-art-xl', weight_name='pixel-art-xl.safetensors', adapter_name='pixelart')
# LoRA LCM — inférence rapide en 8 étapes (essentiel vu le volume à générer sur GPU gratuit limité en temps)
pipe.load_lora_weights('latent-consistency/lcm-lora-sdxl', adapter_name='lcm')
pipe.set_adapters(['pixelart', 'lcm'], adapter_weights=[1.2, 1.0])
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

print('Pipeline chargé.')

In [ ]:
from PIL import Image
from rembg import remove

def generate_asset(name, prompt, category='test', sprite_size=128, remove_bg=True, seed=None):
    """Génère un asset pixel art : rendu SDXL en 1024, downscale nearest-neighbor
    vers sprite_size (technique recommandée par nerijs pour un rendu pixel-perfect),
    puis suppression de fond optionnelle (rembg) pour un PNG transparent."""
    generator = torch.Generator('cuda').manual_seed(seed) if seed is not None else None
    full_prompt = f"pixel art, {prompt}, jeu vidéo, fond uni simple"
    image = pipe(
        prompt=full_prompt,
        num_inference_steps=8,
        guidance_scale=1.5,
        generator=generator,
    ).images[0]

    small = image.resize((sprite_size, sprite_size), Image.NEAREST)

    if remove_bg:
        small = remove(small)

    out_path = f'{OUTPUT}/{category}/{name}.png'
    small.save(out_path)
    return small, out_path

## Lot de test (3 avatars)

But : valider le style pixel art / la cohérence avant de committer sur les ~600 assets (avatars, skins, auras, cadres, effets, animations). Regarde les 3 images générées ci-dessous — si le style ne convient pas (proportions, palette, lisibilité), on ajuste le prompt/les paramètres ici avant d'aller plus loin.

In [ ]:
from IPython.display import display

test_prompts = [
    ('avatar_test_mc', "MC de rap français, casquette, chaîne en or, micro à la main, buste, cadrage frontal"),
    ('avatar_test_dj', "DJ avec casque audio et platines vinyle, buste, cadrage frontal"),
    ('avatar_test_graffeur', "graffeur avec bombe de peinture et bonnet, buste, cadrage frontal"),
]

for name, prompt in test_prompts:
    img, path = generate_asset(name, prompt, category='test', sprite_size=128, seed=42)
    print(name, '->', path)
    display(img)

## Prochaine étape

Une fois ce test validé (style, lisibilité, cohérence), l'étape suivante est un notebook/manifest séparé qui pilote la génération complète à partir d'un fichier JSON listant les ~600 assets (20 avatars × 10 skins, 10 auras, 25 cadres, 30 effets, 50 animations), avec reprise possible en cas de déconnexion (on ne régénère pas ce qui existe déjà dans `output/`).